# Faster R-CNN seeds on Google Colab

Trains the primary model (Faster R-CNN, ResNet-50 FPN v2) for the remaining seeds with exactly the
settings of seed 0 (`--batch 1`), then scores each on the DENTEX test split.

**Before running:** Runtime > Change runtime type > **T4 GPU**. Then Runtime > **Run all**.

- Checkpoints and results are saved to Google Drive (`MyDrive/caries_runs/`), so if Colab disconnects,
  just **Run all** again: finished seeds are skipped and an interrupted one resumes from its last epoch.
- Keep this tab open while it trains (Colab disconnects idle sessions).
- When both seeds are done, download `MyDrive/caries_runs/frcnn_results.zip` (small, no weights) and put
  it in `~/Downloads` on the Mac.

In [1]:
SEEDS = [1, 2]  # seeds to train here

import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout
assert gpu, "No GPU: Runtime > Change runtime type > T4 GPU, then Run all again"
print("GPU:", gpu)
from google.colab import drive
drive.mount("/content/drive")

FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'

In [ ]:
%cd /content
!test -d caries || git clone -q https://github.com/felix-chima/explainable-caries-detection.git caries
%cd /content/caries
!git pull -q && git log --oneline -1

In [ ]:
# DENTEX from Hugging Face (fast inside Google), then the same 70/15/15 split as on the Mac
import shutil, subprocess, sys
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download

if not Path("data/dentex_yolo/images/test").exists():
    raw = Path("data/dentex"); raw.mkdir(parents=True, exist_ok=True)
    need = {"training_data.zip", "validation_data.zip", "validation_triple.json"}
    files = [f for f in HfApi().list_repo_files("ibrahimhamamci/DENTEX", repo_type="dataset") if Path(f).name in need]
    assert len(files) == 3, files
    for f in files:
        print("downloading", f)
        shutil.move(hf_hub_download("ibrahimhamamci/DENTEX", f, repo_type="dataset", local_dir="/content/hf"),
                    raw / Path(f).name)
    out = subprocess.run([sys.executable, "scripts/prepare_dentex.py"], capture_output=True, text=True)
    print(out.stdout[-1500:], out.stderr[-1500:])
    assert "split matches" in out.stdout, "Split differs from the Mac's: do not train, report this"
    for z in raw.glob("*.zip"):
        z.unlink()  # free disk
print("dataset ready:", {s: len(list(Path(f"data/dentex_yolo/images/{s}").glob("*.png"))) for s in ("train", "val", "test")})

In [ ]:
# runs/frcnn and runs/eval live on Google Drive so they survive disconnects
DRIVE = Path("/content/drive/MyDrive/caries_runs")
for sub in ("frcnn", "eval"):
    local, remote = Path("runs") / sub, DRIVE / sub
    remote.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        continue
    if local.exists():  # seed-0 files from git: copy across without overwriting
        for f in local.rglob("*"):
            dst = remote / f.relative_to(local)
            if f.is_file() and not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst)
        shutil.rmtree(local)
    local.parent.mkdir(exist_ok=True)
    local.symlink_to(remote, target_is_directory=True)
print("runs on Drive:", DRIVE)

In [ ]:
def run(cmd, log):
    with open(log, "a") as f:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            f.write(line); f.flush()
            if "arning" not in line:
                print(line, end="", flush=True)
        return p.wait()

for seed in SEEDS:
    name = f"frcnn_s{seed}"
    log = Path(f"runs/frcnn/{name}.log")
    if log.exists() and any(l.startswith("done") for l in log.read_text().splitlines()):
        print(name, "already trained")
    else:
        resume = Path(f"runs/frcnn/{name}/weights/last.pt").exists()
        print(name, "resuming" if resume else "starting")
        cmd = [sys.executable, "scripts/train_frcnn.py", "--name", name, "--seed", str(seed), "--batch", "1"]
        assert run(cmd + (["--resume"] if resume else []), log) == 0, f"{name} failed, see {log}"
    if not Path(f"runs/eval/{name}_test.json").exists():
        print("evaluating", name)
        assert run([sys.executable, "scripts/evaluate.py", "frcnn", f"runs/frcnn/{name}/weights/best.pt",
                    "--name", name], log) == 0
    print(open(f"runs/eval/{name}_test.json").read())

In [ ]:
# small results bundle for the Mac (no weights)
!cd runs && rm -f {DRIVE}/frcnn_results.zip && zip -qr {DRIVE}/frcnn_results.zip frcnn eval -x "frcnn/*/weights/*" && ls -la {DRIVE}/frcnn_results.zip
print("Download MyDrive/caries_runs/frcnn_results.zip and put it in ~/Downloads on the Mac.")

In [3]:
!pip install --force-reinstall torch torchvision torchaudio --index-url https://pytorch.org

Looking in indexes: https://pytorch.org
ERROR: Could not find a version that satisfies the requirement torch (from versions: none)
ERROR: No matching distribution found for torch


In [2]:
# YOLOv8s seed 2 on Colab: self-contained (same settings as seed 0 on the Mac)
import os, sys, shutil, subprocess
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

# 1. repo
os.chdir("/content")
if not Path("caries").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/felix-chima/explainable-caries-detection.git", "caries"], check=True)
os.chdir("/content/caries")
subprocess.run(["git", "pull", "-q"])

# 2. dataset (skipped if already built in this session)
if not Path("data/dentex_yolo/images/test").exists():
    from huggingface_hub import HfApi, hf_hub_download
    raw = Path("data/dentex"); raw.mkdir(parents=True, exist_ok=True)
    need = {"training_data.zip", "validation_data.zip", "validation_triple.json"}
    for f in [f for f in HfApi().list_repo_files("ibrahimhamamci/DENTEX", repo_type="dataset") if Path(f).name in need]:
        print("downloading", f)
        shutil.move(hf_hub_download("ibrahimhamamci/DENTEX", f, repo_type="dataset", local_dir="/content/hf"), raw / Path(f).name)
    out = subprocess.run([sys.executable, "scripts/prepare_dentex.py"], capture_output=True, text=True)
    print(out.stdout[-600:], out.stderr[-600:])
    assert "split matches" in out.stdout, "Split differs from the Mac's: stop and report this"
    for z in raw.glob("*.zip"):
        z.unlink()
print("dataset:", {s: len(list(Path(f"data/dentex_yolo/images/{s}").glob("*.png"))) for s in ("train", "val", "test")})

# 3. results folders on Drive
DRIVE = Path("/content/drive/MyDrive/caries_runs")
for sub in ("frcnn", "eval"):
    local, remote = Path("runs") / sub, DRIVE / sub
    remote.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        continue
    if local.exists():
        for f in local.rglob("*"):
            dst = remote / f.relative_to(local)
            if f.is_file() and not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(f, dst)
        shutil.rmtree(local)
    local.parent.mkdir(exist_ok=True)
    local.symlink_to(remote, target_is_directory=True)

def run(cmd, log):
    with open(log, "a") as fh:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            fh.write(line); fh.flush()
            if "arning" not in line:
                print(line, end="", flush=True)
        return p.wait()

# 4. train YOLOv8s seed 2 (resumes if interrupted, skips if done)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "ultralytics==8.4.173"], check=True)
YOLO = str(Path(sys.executable).parent / "yolo")
seed, name = 2, "yolov8s_s2"
YDIR = DRIVE / "detect"
run_dir = YDIR / f"yolo_s{seed}"
res = run_dir / "results.csv"
log = DRIVE / "detect" / f"{name}.log"
log.parent.mkdir(parents=True, exist_ok=True)
if res.exists() and len(res.read_text().splitlines()) - 1 >= 100:
    print(name, "already trained")
elif (run_dir / "weights/last.pt").exists():
    print(name, "resuming")
    assert run(["yolo", "detect", "train", "resume", f"model={run_dir}/weights/last.pt"], log) == 0
else:
    print(name, "starting")
    assert run(["yolo", "detect", "train", "model=yolov8s.pt", "data=caries.yaml", "epochs=100", "patience=100",
                "batch=16", "imgsz=640", "device=0", "workers=2", f"seed={seed}", "deterministic=True",
                f"project={YDIR}", f"name=yolo_s{seed}", "exist_ok=True"], log) == 0

# 5. score on the test split, then bundle results for the Mac
if not Path(f"runs/eval/{name}_test.json").exists():
    assert run([sys.executable, "scripts/evaluate.py", "yolo", str(run_dir / "weights/best.pt"), "--name", name], log) == 0
print(open(f"runs/eval/{name}_test.json").read())
subprocess.run(f'cd "{DRIVE}" && rm -f frcnn_results.zip && zip -qr frcnn_results.zip frcnn eval detect -x "*/weights/*" "*.jpg" && ls -la frcnn_results.zip', shell=True)
print("Done. Download MyDrive/caries_runs/frcnn_results.zip into ~/Downloads on the Mac.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
dataset: {'train': 528, 'val': 113, 'test': 114}
yolov8s_s2 starting
New https://pypi.org/project/ultralytics/8.4.175 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.173 🚀 Python-3.13.16 torch-2.11.0+cpu 
Traceback (most recent call last):
  File "/usr/local/bin/yolo", line 8, in <module>
    sys.exit(entrypoint())
             ~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/ultralytics/cfg/__init__.py", line 1187, in entrypoint
    getattr(model, mode)(**overrides)  # default args from model
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/ultralytics/engine/model.py", line 884, in train
    self.trainer = (trainer or self._smart_load("trainer"))(overrides=args, _callbacks=self.callbacks)
                   ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

AssertionError: 